# Measurement error — latent endogenous outcomes

Declare latent variables with `observed=...`. They still have response types and
may appear in queries, but do **not** index the observational table $P$.


In [ ]:
from partialid import Assumptions, CausalDiagram, Query, Variables, bound

# Latent Y; observe misclassified Y2.  X → Y → Y2, confounding Y↔Y2.
V = Variables.from_binary("X", "Y", "Y2", observed=["X", "Y2"])
print("all names:", V.names)
print("observed:", V.observed)
print("latent:", [n for n in V.names if n not in V.observed])


## Diagram + monotone measurement channel


In [ ]:
g = CausalDiagram.from_edges(
    ["X", "Y", "Y2"],
    directed=[("X", "Y"), ("Y", "Y2")],
    bidirected=[("Y", "Y2")],
    maximal_confounding=False,
)
# P is on observed columns (X, Y2) only
P = {(0, 0): 0.30, (0, 1): 0.20, (1, 0): 0.15, (1, 1): 0.35}
Q = Query.ate(outcome="Y", treatment="X", y=1, t1=1, t0=0)
a = Assumptions.from_diagram(g) & Assumptions.mtr("Y2", "Y")

r = bound(V, P, Q, a)
print(r.summary(Q))
print(f"tight={r.tight}  method={r.method}")


## Without MTR the ATE interval is wider


In [ ]:
r0 = bound(V, P, Q, Assumptions.from_diagram(g))
print(f"diagram only:  [{r0.lower:.4f}, {r0.upper:.4f}] w={r0.width:.4f}")
print(f"+ MTR(Y2|Y):   [{r.lower:.4f}, {r.upper:.4f}] w={r.width:.4f}")


## Interventional query on the latent outcome


In [ ]:
Qi = Query.interventional(outcome="Y", do={"X": 1}, value=1)
ri = bound(V, P, Qi, a)
print(ri.summary(Qi))
